In [36]:
import pandas as pd
import numpy as np
import torch
import plotly.graph_objects as go

from src.utils import generate_mask_tensor
from src.embedding import embed
from src.gp_ccm import GP_ccm_sig
from src.iaaft import surrogates

In [37]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

Using device: cuda



In [38]:
# Set length of timeseries
N_length = torch.tensor([400])

# Initialise values at t = 0
z = torch.tensor([0.2])
x = torch.tensor([0.3])
y = torch.tensor([0.4])

# Autoregressive function
for t in range(N_length - 1):
    
    z_next = z[t] * (3.9 - (3.9 * z[t]))
    x_next = x[t] * (3.8 - (3.8 * x[t]) - (0.2 * z[t]))
    y_next = y[t] * (3.6 - (3.6 * y[t]) - (0.2 * z[t]))

    z = torch.concat((z, z_next.unsqueeze(0)))
    x = torch.concat((x, x_next.unsqueeze(0)))
    y = torch.concat((y, y_next.unsqueeze(0)))
            
# Normalising step
z_norm = z.sub(z.mean(dim = -1).unsqueeze(-1)).div(z.std(dim = -1).unsqueeze(-1))
x_norm = x.sub(x.mean(dim = -1).unsqueeze(-1)).div(x.std(dim = -1).unsqueeze(-1))
y_norm = y.sub(y.mean(dim = -1).unsqueeze(-1)).div(y.std(dim = -1).unsqueeze(-1))

This still works:

In [39]:
# Set length of timeseries
N_length = torch.tensor([400])

# Initialise values at t = 0
z = torch.tensor([0.2])
x = torch.tensor([0.3])
y = torch.tensor([0.4])

# Autoregressive function
for t in range(N_length - 1):
    
    z_next = z[t] * (3.9 - (3.9 * z[t]))
    x_next = x[t] * (3.8 - (3.8 * x[t]) - (0.3 * z[t]))
    y_next = y[t] * (3.6 - (3.6 * y[t]) - (0.2 * z[t]))

    z = torch.concat((z, z_next.unsqueeze(0)))
    x = torch.concat((x, x_next.unsqueeze(0)))
    y = torch.concat((y, y_next.unsqueeze(0)))
            
# Normalising step
z_norm = z.sub(z.mean(dim = -1).unsqueeze(-1)).div(z.std(dim = -1).unsqueeze(-1))
x_norm = x.sub(x.mean(dim = -1).unsqueeze(-1)).div(x.std(dim = -1).unsqueeze(-1))
y_norm = y.sub(y.mean(dim = -1).unsqueeze(-1)).div(y.std(dim = -1).unsqueeze(-1))

In [40]:
fig = go.Figure()

fig.add_trace(go.Scatter(x = torch.arange(0, z_norm.shape[0]), y = z_norm,
                    mode = 'lines+markers',
                    name = 'Z'))

fig.add_trace(go.Scatter(x = torch.arange(0, z_norm.shape[0]), y = x_norm,
                    mode = 'lines+markers',
                    name = 'X'))

fig.add_trace(go.Scatter(x = torch.arange(0, z_norm.shape[0]), y = y_norm,
                    mode = 'lines+markers',
                    name = 'Y'))

fig.update_layout(title = 'Confounding time series"',
                   xaxis_title = 'epoch',
                   yaxis_title = 'loss')

fig.show()

# X -> Y

In [41]:
shifts = torch.arange(- 4, 4 + 1, 1)
print(shifts)

# fix filter
k = 2 # 2 best
sig_filter = torch.ones(size = (k, )).to(device)

noise_scalar = torch.tensor([0.05], device = device)

rho_s = torch.zeros(size = (shifts.shape[0], 2))

for i, s in enumerate(shifts):
    print(s.item())

    y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = y_norm.to(device), # Testing X -> Y
                           x = x_norm.to(device),
                           max_pos_offset = s, 
                           device = device)
    
    # N now changes slightly
    N = y_embeddings.shape[0]
    E = y_embeddings.shape[1]

    l_train_masks = generate_mask_tensor(N, 100)

    rho_l = torch.empty(size = (1, 0)).to(device)

    for l in range(N):   
        rho, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                x_train = x_gt[l_train_masks[l]].to(device),
                x_test = x_gt[ ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.2, # 0.2 is good
                device = device)
                
        rho_l = torch.concat((rho_l, rho.unsqueeze(0).unsqueeze(0)), dim = 1)
     
    rho_s[i, 0] = rho_l.mean()
    rho_s[i, 1] = rho_l.std()


fig = go.Figure()

fig.add_trace(go.Scatter(x = -shifts, y = rho_s[:, 0], # reversing the meaning of x
                    mode = 'lines+markers',
                    name = 'mean',
                    line_color = "#C00000"))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] + rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines'
))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] - rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines',
    fillcolor = 'rgba(243, 176, 210, 0.3)',
    fill = 'tonexty'
))

fig.add_vline(x = 0.)
fig.add_vline(x = -(k-1), line_dash = "dash")

fig.update_layout(title = 'Cross-mapping skill for X -> Y',
                   xaxis_title = 'shift',
                   yaxis_title = 'rho')

fig.update_layout(template = "plotly_white")
fig.update_layout(font_family = "Lato")

fig.show()

tensor([-4, -3, -2, -1,  0,  1,  2,  3,  4])
-4
-3
-2
-1
0
1
2
3
4


# Y -> X

In [42]:
shifts = torch.arange(- 4, 4 + 1, 1)
print(shifts)

# fix filter
k = 2 # 2 best
sig_filter = torch.ones(size = (k, )).to(device)

noise_scalar = torch.tensor([0.05], device = device)

rho_s = torch.zeros(size = (shifts.shape[0], 2))

for i, s in enumerate(shifts):
    print(s.item())

    y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = x_norm.to(device), # Testing Y -> X
                           x = y_norm.to(device),
                           max_pos_offset = s, 
                           device = device)
    
    # N now changes slightly
    N = y_embeddings.shape[0]
    E = y_embeddings.shape[1]

    l_train_masks = generate_mask_tensor(N, 100)

    rho_l = torch.empty(size = (1, 0)).to(device)

    for l in range(N):   
        rho, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                x_train = x_gt[l_train_masks[l]].to(device),
                x_test = x_gt[ ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.2, # 0.2 is good
                device = device)
                
        rho_l = torch.concat((rho_l, rho.unsqueeze(0).unsqueeze(0)), dim = 1)
     
    rho_s[i, 0] = rho_l.mean()
    rho_s[i, 1] = rho_l.std()


fig = go.Figure()

fig.add_trace(go.Scatter(x = -shifts, y = rho_s[:, 0], # reversing the meaning of x
                    mode = 'lines+markers',
                    name = 'mean',
                    line_color = "#C00000"))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] + rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines'
))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] - rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines',
    fillcolor = 'rgba(243, 176, 210, 0.3)',
    fill = 'tonexty'
))

fig.add_vline(x = 0.)
fig.add_vline(x = -(k-1), line_dash = "dash")

fig.update_layout(title = 'Cross-mapping skill for Y -> X',
                   xaxis_title = 'shift',
                   yaxis_title = 'rho')

fig.update_layout(template = "plotly_white")
fig.update_layout(font_family = "Lato")

fig.show()

tensor([-4, -3, -2, -1,  0,  1,  2,  3,  4])
-4
-3
-2
-1
0
1
2
3
4


# Z -> X

In [43]:
shifts = torch.arange(- 4, 4 + 1, 1)
print(shifts)

# fix filter
k = 2 # 2 best
sig_filter = torch.ones(size = (k, )).to(device)

noise_scalar = torch.tensor([0.05], device = device)

rho_s = torch.zeros(size = (shifts.shape[0], 2))

for i, s in enumerate(shifts):
    print(s.item())

    y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = x_norm.to(device), # Testing Z -> X
                           x = z_norm.to(device),
                           max_pos_offset = s, 
                           device = device)
    
    # N now changes slightly
    N = y_embeddings.shape[0]
    E = y_embeddings.shape[1]

    l_train_masks = generate_mask_tensor(N, 100)

    rho_l = torch.empty(size = (1, 0)).to(device)

    for l in range(N):   
        rho, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                x_train = x_gt[l_train_masks[l]].to(device),
                x_test = x_gt[ ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.2, # 0.2 is good
                device = device)
                
        rho_l = torch.concat((rho_l, rho.unsqueeze(0).unsqueeze(0)), dim = 1)
     
    rho_s[i, 0] = rho_l.mean()
    rho_s[i, 1] = rho_l.std()


fig = go.Figure()

fig.add_trace(go.Scatter(x = -shifts, y = rho_s[:, 0], # reversing the meaning of x
                    mode = 'lines+markers',
                    name = 'mean',
                    line_color = "#C00000"))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] + rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines'
))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] - rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines',
    fillcolor = 'rgba(243, 176, 210, 0.3)',
    fill = 'tonexty'
))

fig.add_vline(x = 0.)
fig.add_vline(x = -(k-1), line_dash = "dash")

fig.update_layout(title = 'Cross-mapping skill for Z -> X',
                   xaxis_title = 'shift',
                   yaxis_title = 'rho')

fig.update_layout(template = "plotly_white")
fig.update_layout(font_family = "Lato")

fig.show()

tensor([-4, -3, -2, -1,  0,  1,  2,  3,  4])
-4
-3
-2
-1
0
1
2
3
4


# Z -> Y

In [44]:
shifts = torch.arange(-4, 4 + 1, 1)
print(shifts)

# fix filter
k = 2 # 2 best
sig_filter = torch.ones(size = (k, )).to(device)

noise_scalar = torch.tensor([0.05], device = device)

rho_s = torch.zeros(size = (shifts.shape[0], 2))

for i, s in enumerate(shifts):
    print(s.item())

    y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = y_norm.to(device), # Testing Z -> Y
                           x = z_norm.to(device),
                           max_pos_offset = s, 
                           device = device)
    
    # N now changes slightly
    N = y_embeddings.shape[0]
    E = y_embeddings.shape[1]

    l_train_masks = generate_mask_tensor(N, 100)

    rho_l = torch.empty(size = (1, 0)).to(device)

    for l in range(N):   
        rho, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                x_train = x_gt[l_train_masks[l]].to(device),
                x_test = x_gt[ ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.2, # 0.2 is good
                device = device)
                
        rho_l = torch.concat((rho_l, rho.unsqueeze(0).unsqueeze(0)), dim = 1)
     
    rho_s[i, 0] = rho_l.mean()
    rho_s[i, 1] = rho_l.std()


fig = go.Figure()

fig.add_trace(go.Scatter(x = -shifts, y = rho_s[:, 0], # reversing the meaning of x
                    mode = 'lines+markers',
                    name = 'mean',
                    line_color = "#C00000"))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] + rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines'
))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] - rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines',
    fillcolor = 'rgba(243, 176, 210, 0.3)',
    fill = 'tonexty'
))

fig.add_vline(x = 0.)
fig.add_vline(x = -(k-1), line_dash = "dash")

fig.update_layout(title = 'Cross-mapping skill for Z -> Y',
                   xaxis_title = 'shift',
                   yaxis_title = 'rho')

fig.update_layout(template = "plotly_white")
fig.update_layout(font_family = "Lato")

fig.show()

tensor([-4, -3, -2, -1,  0,  1,  2,  3,  4])
-4
-3
-2
-1
0
1
2
3
4


# X -> Z

In [45]:
shifts = torch.arange(- 4, 4 + 1, 1)
print(shifts)

# fix filter
k = 2 # 2 best
sig_filter = torch.ones(size = (k, )).to(device)

noise_scalar = torch.tensor([0.05], device = device)

rho_s = torch.zeros(size = (shifts.shape[0], 2))

for i, s in enumerate(shifts):
    print(s.item())

    y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = z_norm.to(device), # Testing Z -> X
                           x = x_norm.to(device),
                           max_pos_offset = s, 
                           device = device)
    
    # N now changes slightly
    N = y_embeddings.shape[0]
    E = y_embeddings.shape[1]

    l_train_masks = generate_mask_tensor(N, 100)

    rho_l = torch.empty(size = (1, 0)).to(device)

    for l in range(N):   
        rho, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                x_train = x_gt[l_train_masks[l]].to(device),
                x_test = x_gt[ ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.2, # 0.2 is good
                device = device)
                
        rho_l = torch.concat((rho_l, rho.unsqueeze(0).unsqueeze(0)), dim = 1)
     
    rho_s[i, 0] = rho_l.mean()
    rho_s[i, 1] = rho_l.std()


fig = go.Figure()

fig.add_trace(go.Scatter(x = -shifts, y = rho_s[:, 0], # reversing the meaning of x
                    mode = 'lines+markers',
                    name = 'mean',
                    line_color = "#C00000"))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] + rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines'
))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] - rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines',
    fillcolor = 'rgba(243, 176, 210, 0.3)',
    fill = 'tonexty'
))

fig.add_vline(x = 0.)
fig.add_vline(x = -(k-1), line_dash = "dash")

fig.update_layout(title = 'Cross-mapping skill for Z -> X',
                   xaxis_title = 'shift',
                   yaxis_title = 'rho')

fig.update_layout(template = "plotly_white")
fig.update_layout(font_family = "Lato")

fig.show()

tensor([-4, -3, -2, -1,  0,  1,  2,  3,  4])
-4
-3
-2
-1
0
1
2
3
4


# Y -> Z

In [46]:
shifts = torch.arange(- 4, 4 + 1, 1)
print(shifts)

# fix filter
k = 2 # 2 best
sig_filter = torch.ones(size = (k, )).to(device)

noise_scalar = torch.tensor([0.05], device = device)

rho_s = torch.zeros(size = (shifts.shape[0], 2))

for i, s in enumerate(shifts):
    print(s.item())

    y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = z_norm.to(device), # Testing Y -> Z 
                           x = y_norm.to(device),
                           max_pos_offset = s, 
                           device = device)
    
    # N now changes slightly
    N = y_embeddings.shape[0]
    E = y_embeddings.shape[1]

    l_train_masks = generate_mask_tensor(N, 100)

    rho_l = torch.empty(size = (1, 0)).to(device)

    for l in range(N):   
        rho, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                x_train = x_gt[l_train_masks[l]].to(device),
                x_test = x_gt[ ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.2, # 0.2 is good
                device = device)
                
        rho_l = torch.concat((rho_l, rho.unsqueeze(0).unsqueeze(0)), dim = 1)
     
    rho_s[i, 0] = rho_l.mean()
    rho_s[i, 1] = rho_l.std()


fig = go.Figure()

fig.add_trace(go.Scatter(x = -shifts, y = rho_s[:, 0], # reversing the meaning of x
                    mode = 'lines+markers',
                    name = 'mean',
                    line_color = "#C00000"))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] + rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines'
))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] - rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines',
    fillcolor = 'rgba(243, 176, 210, 0.3)',
    fill = 'tonexty'
))

fig.add_vline(x = 0.)
fig.add_vline(x = -(k-1), line_dash = "dash")

fig.update_layout(title = 'Cross-mapping skill for Y -> Z',
                   xaxis_title = 'shift',
                   yaxis_title = 'rho')

fig.update_layout(template = "plotly_white")
fig.update_layout(font_family = "Lato")

fig.show()

tensor([-4, -3, -2, -1,  0,  1,  2,  3,  4])
-4
-3
-2
-1
0
1
2
3
4
